[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/25_flash_attention_solution.ipynb)

# 🔴 Solution: Flash Attention (Tiled)（参考版）

## 📋 题目描述

**难度：** Hard

实现分块（Flash）注意力与在线 softmax。

Flash 注意力将 Q/K/V 分块处理以减少内存使用，利用在线 softmax 技巧在分块间保持数值正确性。

**签名:** `flash_attention(Q, K, V, block_size=32) -> Tensor`

**参数:**
- `Q`, `K`, `V` — 输入张量 (B, S, D)
- `block_size` — 分块大小

**返回:** 注意力输出 (B, S, D)，与标准注意力数值一致

**约束:**
- 必须与标准 softmax 注意力数值匹配
- 处理非对齐序列长度（S 不能被 block_size 整除）
- 结果不受 block_size 选择影响

**提示：** 分块处理 Q。对每个 Q 块，遍历 K/V 块：
1. `block_max = scores.max(dim=-1)` → `new_max = max(row_max, block_max)`
2. `correction = exp(row_max - new_max)` → 重新缩放 `acc` 和 `row_sum`
3. `acc += exp(scores - new_max) @ V_block`
4. `output = acc / row_sum`


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math


In [ ]:
# ✅ SOLUTION

def flash_attention(Q, K, V, block_size=32):
    B, S, D = Q.shape
    output = torch.zeros_like(Q)

    # ---- Step 1: 外层循环 — 遍历 Q 的分块 ----
    # 将 Q 按 block_size 分块，每个 Q 块独立处理
    # 这样每个 Q 块只需要维护一个 d_k 维的累加器，而不是 S×S 的注意力矩阵
    for i in range(0, S, block_size):
        qi = Q[:, i:i+block_size]  # (B, bs_q, D)
        bs_q = qi.shape[1]  # 最后一块可能不足 block_size

        # ---- Step 2: 初始化该 Q 块的在线状态 ----
        # row_max: 该 Q 块中每个 query 的当前最大分数（用于在线 softmax）
        # row_sum: 该 Q 块中每个 query 的 exp 分数之和
        # acc: 该 Q 块中每个 query 的加权 value 累加器
        row_max = torch.full((B, bs_q, 1), float('-inf'), device=Q.device)
        row_sum = torch.zeros(B, bs_q, 1, device=Q.device)
        acc = torch.zeros(B, bs_q, D, device=Q.device)

        # ---- Step 3: 内层循环 — 遍历 K/V 的分块 ----
        # 对于每个 Q 块，需要遍历所有 K/V 块
        # 这就是 O(S²) 复杂度，但内存是 O(S) 而非 O(S²)
        for j in range(0, S, block_size):
            kj = K[:, j:j+block_size]  # (B, bs_kv, D)
            vj = V[:, j:j+block_size]  # (B, bs_kv, D)

            # ---- Step 4: 计算当前块的注意力分数 ----
            # scores = Q_i @ K_j^T / sqrt(D)
            # shape: (B, bs_q, bs_kv)
            scores = torch.bmm(qi, kj.transpose(1, 2)) / math.sqrt(D)

            # ---- Step 5: 在线 softmax 更新 ----
            # 核心思想：softmax 需要全局 max 和 sum，但我们分块处理
            # 所以需要在每块之后"修正"之前的累加结果
            #
            # 5a. 找当前块的最大值
            block_max = scores.max(dim=-1, keepdim=True).values  # (B, bs_q, 1)

            # 5b. 更新全局最大值
            new_max = torch.maximum(row_max, block_max)  # (B, bs_q, 1)

            # 5c. 计算修正系数
            # 之前累加的 acc 和 row_sum 是基于旧的 row_max 计算的
            # 现在最大值变了，需要用 exp(旧max - 新max) 修正之前的累加值
            correction = torch.exp(row_max - new_max)  # (B, bs_q, 1)

            # ---- Step 6: 计算当前块的 exp(scores - new_max) ----
            # 用 new_max 做数值稳定的 exp
            exp_scores = torch.exp(scores - new_max)  # (B, bs_q, bs_kv)

            # ---- Step 7: 更新累加器 ----
            # acc = 旧acc × 修正系数 + 当前块的 exp_scores @ V
            # row_sum = 旧row_sum × 修正系数 + 当前块的 exp_scores 之和
            acc = acc * correction + torch.bmm(exp_scores, vj)
            row_sum = row_sum * correction + exp_scores.sum(dim=-1, keepdim=True)
            row_max = new_max

        # ---- Step 8: 归一化 ----
        # output = acc / row_sum，等价于 softmax(scores) @ V
        output[:, i:i+block_size] = acc / row_sum

    return output


In [ ]:
Q = torch.randn(1, 16, 32)
K = torch.randn(1, 16, 32)
V = torch.randn(1, 16, 32)
out_flash = flash_attention(Q, K, V, block_size=4)
out_std = torch.softmax(Q @ K.T / math.sqrt(32), dim=-1) @ V
print(f'Match: {torch.allclose(out_flash, out_std, atol=1e-5)}')


In [ ]:
from torch_judge import check
check('flash_attention')


## 📝 核心思路总结

1. **分块处理**：Q 和 K/V 都按 block_size 分块，避免存储完整 S×S 注意力矩阵
2. **在线 softmax**：维护 row_max 和 row_sum，每块后用修正系数更新
3. **修正系数**：`correction = exp(旧max - 新max)`，确保之前累加值的数值正确性
4. **内存优势**：O(S) vs O(S²)，对长序列至关重要
